In [1]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers

In [4]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import tool

from langgraph.prebuilt import create_react_agent

In [6]:
# ----------------------------
# Configure Gemini API
# ----------------------------

# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [7]:
# -----------------------------
# Embedding model
# -----------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# -----------------------------
# Knowledge base documents
# -----------------------------

documents = [

Document(page_content="""
Solar energy converts sunlight into electricity using photovoltaic panels.
It is a renewable energy source that reduces carbon emissions and can be
installed on rooftops or in large solar farms.
"""),

Document(page_content="""
Wind energy is generated by turbines that convert wind motion into electricity.
Large wind farms are built in areas with strong wind such as coastal or offshore regions.
"""),

Document(page_content="""
Hydropower uses flowing water to spin turbines connected to generators.
Dams and reservoirs help control water flow and generate consistent electricity.
"""),

Document(page_content="""
Geothermal energy uses underground heat from the Earth to generate electricity.
It is highly reliable because it does not depend on weather conditions.
""")
]


/tmp/ipykernel_971/2332670551.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:


# -----------------------------
# Vector database
# -----------------------------

vector_db = FAISS.from_documents(documents, embedding_model)


# -----------------------------
# Tool: vector search
# -----------------------------

@tool
def search_knowledge(query: str) -> str:
    """Search renewable energy knowledge base."""
    results = vector_db.similarity_search(query, k=2)
    return "\n".join([doc.page_content for doc in results])


tools = [search_knowledge]


# -----------------------------
# Gemini LLM
# -----------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [9]:
# -----------------------------
# Create agent
# -----------------------------

agent = create_react_agent(
    llm,
    tools
)

/tmp/ipykernel_971/2974466127.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [10]:
# -----------------------------
# Run agent
# -----------------------------

while True:

    question = input("\nAsk a question (type exit): ")

    if question.lower() == "exit":
        break

    result = agent.invoke({
        "messages": [("user", question)]
    })

    print("\nAnswer:")
    print(result["messages"][-1].content)


Ask a question (type exit): What are the benefits of solar energy?

Answer:
[{'type': 'text', 'text': 'Solar energy is a renewable energy source that converts sunlight into electricity using photovoltaic panels. It helps in reducing carbon emissions and can be installed in various locations, such as rooftops or large solar farms.', 'extras': {'signature': 'CpgFAb4+9vt/ovwpIIoYN7oM2MFFhuJneCKcBhobzP3B5hd9KLt3zQEQlNHsc1z1MeA6N2vuPPDQ27V/H+sEx9IC9AZEGQogA1mGjxveok83HkF54EYAImzDyfQdBLjoWERpqDdEzLqqzx4K3MDQ4AZmovyeiLKfV1Cu8uRYd8+BMqA60DXNs3S6FlOzjD70PbNUi1bkjll9ROYCd55rrinrh9NnSPL8RCL2dEChLOoQ2tEn5P0MofeQXsutsgOxJjUAjMmd88Nu/ES1oW17fdtlGXEfnIhmTb+inOIdlEGCMlXjx/9lcihWmaGuxnST+L8zC6b/JWPtAorcXDX1zDVMN/iMR7nAhue/oESJ59OWJzxdZ76bMd++aBLXjbx54cdhZtIrztUB7snqTYheR8F6jyRmumgofNp+PE3NBN4X71LujMflCxQw5Jq/UUME6DOfjghlHggdQfLN6a8qG+NSS2ds/NL2D6sfb7NL0x+f/AKfg81j8A/7Qi99d9TI8toVrCgxh7/XZmgqG7JVTdDqYejRY+z6ZnWZz0zjz5r4uRANqnt8RKSis6TEeGynUkQ1mf/NZXqD51ksOay0LZwG07Ql54Oa/H6H98fL3vA0dNo1Mn5XtEYEqlvoCrvi